# PeakDeviation

Для кожного тікера з `final.parquet`: скільки разів і наскільки сильно Stack%
відхиляється від референс-клоуза, окремо для додатних і від'ємних відхилень —
і яке з цих пікових значень найчастіше повторюється (типове, а не разове,
відхилення для цього стака).


In [ ]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# Import basic modules
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow.parquet as pq

import warnings
warnings.filterwarnings("ignore")
# pip install openpyxl


In [ ]:
def _resolve_orion_paths(strategy_code: str):
    """Same convention ArbitRage.ipynb/Reversal.ipynb use to find final.parquet
    without hardcoding a machine-specific path."""
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME (recommended).")
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


def _resolve_axion_tape_root() -> Path:
    """Root of the live bridge's own per-minute tape state (NOT a network call --
    this is a plain filesystem read of files the bridge already wrote to disk)."""
    env = os.environ.get("AXION_TAPE_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    appdata = os.environ.get("APPDATA")
    if not appdata:
        raise RuntimeError("Cannot locate Axion state dir. Set AXION_TAPE_ROOT.")
    return Path(appdata) / "Axion" / "tape" / "intraday" / "v2"


FINAL_PATH, OUT_DIR = _resolve_orion_paths("peakdev")
TAPE_ROOT = _resolve_axion_tape_root()

print("Using FINAL parquet:", FINAL_PATH)
print("Using Axion tape root (ETF/lstcls):", TAPE_ROOT)
print("Output dir:", OUT_DIR)


## Подія (event)

Подія народжується, коли `|Stack%| >= start_thr` (0.5%) — сторона (плюс/мінус)
фіксується на народженні. Подія лишається АКТИВНОЮ, поки `|Stack%|` не
опуститься до `<= exit_thr` (0.1%) — повна нормалізація; проміжна зона
(0.1%..0.5%) НЕ закриває подію, це той самий, ще не нормалізований епізод
(точно як у `ArbitRage.ipynb`: `dev_thr` старт / `norm_thr` вихід,
`close_on_exit=True`). Щойно нормалізувалась — подія закрита; наступний
перетин 0.5% — вже НОВА подія (re-birth). Подія, що не встигла
нормалізуватись до кінця дня, закривається на межі дня (активна подія не
переходить через день) і рахується як НЕнормалізована. Пік події —
максимальне `|Stack%|`, досягнуте всередині неї (окрема характеристика, не
впливає на рейтинг). Подія, що СТАРТУЄ всередині `03:58-04:02` чи
`07:58-08:02` (відомі проблемні хвилини даних — той самий `ignore_windows`,
що й у `ArbitRage.ipynb`), відкидається цілком.

**Рейтинг тікера НЕ дивиться на пік** — це частка його подій, які СПРАВДІ
дозвужились до 0.1% (а не були обрізані кінцем дня, так і лишившись
відкритими): `рейтинг = (подій, що нормалізувались) / total`. Тотал —
загальна кількість подій (обидва знаки). Позитивне/негативне пікове — окремі
колонки, типове (найчастіше, модальне) значення піку серед подій свого
знаку.

In [ ]:
def _modal_value(values, step: float = 0.1, min_n: int = 3) -> float:
    """Значення, яке серія показує НАЙЧАСТІШЕ (мода), забінене з кроком `step`;
    що повертається — СЕРЕДНЄ сирих значень усередині виграшного біну (а не
    його межа), щоб число не було зміщене квантуванням. NaN, якщо readings
    менше min_n (пара відліків — не статистика, а відлуння)."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if v.size < min_n:
        return float("nan")
    bins = np.round(v / step) * step
    uniq, counts = np.unique(bins, return_counts=True)
    best = uniq[np.argmax(counts)]
    in_bin = np.abs(bins - best) < 1e-9
    return float(np.mean(v[in_bin]))


def detect_events(
    df: pd.DataFrame,
    start_thr: float = 0.5,
    exit_thr: float = 0.1,
    ignore_windows=(("03:58", "04:02"), ("07:58", "08:02")),
) -> pd.DataFrame:
    """Подія народжується, коли |Stack%| >= start_thr (сторона фіксується на
    народженні). Лишається АКТИВНОЮ, поки |Stack%| не опуститься до
    <= exit_thr (повна нормалізація) -- проміжна зона (exit_thr, start_thr) НЕ
    закриває подію, це той самий, ще не нормалізований епізод. Щойно
    нормалізувалась -- подія закрита; наступний перетин start_thr -- вже нова
    подія (re-birth), як у ArbitRage (dev_thr=старт / norm_thr=вихід,
    close_on_exit=True). Кожен (ticker, date) стартує з чистого "неактивного"
    стану -- активна подія не переходить через межу дня; недонормалізована
    подія закривається на цій межі. Повертає: ticker, date, side (+1/-1),
    start_dt, peak_abs."""
    d = df.dropna(subset=["Stack%", "ticker", "date", "dt"]).copy()
    d = d.sort_values(["ticker", "date", "dt"], kind="stable").reset_index(drop=True)

    x = pd.to_numeric(d["Stack%"], errors="coerce").to_numpy()
    ax = np.abs(x)

    tkr = d["ticker"].to_numpy(dtype=object)
    dte = d["date"].to_numpy(dtype=object)

    new_grp = np.empty(len(d), dtype=bool)
    new_grp[0] = True
    if len(d) > 1:
        new_grp[1:] = (tkr[1:] != tkr[:-1]) | (dte[1:] != dte[:-1])
    grp_id = np.cumsum(new_grp)

    # +1 = active (born), 0 = fully normalized, NaN = dead zone (holds previous state)
    state = np.where(ax >= start_thr, 1.0, np.where(ax <= exit_thr, 0.0, np.nan))
    seed_mask = new_grp & np.isnan(state)  # a fresh day never inherits an active event
    state[seed_mask] = 0.0

    active = pd.Series(state).groupby(grp_id, sort=False).ffill().to_numpy()
    active = (active == 1.0)

    sign = np.sign(x).astype(np.int8)
    prev_active = np.empty(len(d), dtype=bool)
    prev_active[0] = False
    prev_active[1:] = active[:-1]
    prev_sign = np.empty(len(d), dtype=np.int8)
    prev_sign[0] = 0
    prev_sign[1:] = sign[:-1]

    # while active, a sign flip without ever fully normalizing (a jump straight
    # past the dead zone in one tick) still starts a fresh event
    sign_flip_while_active = active & prev_active & (sign != prev_sign)
    new_event_boundary = new_grp | (active != prev_active) | sign_flip_while_active
    event_id = np.cumsum(new_event_boundary)

    d["_active"] = active
    d["_event"] = event_id

    inside = d[d["_active"]]
    if inside.empty:
        return pd.DataFrame(columns=["ticker", "date", "side", "start_dt", "peak_abs", "normalized"])

    last_of_grp = np.empty(len(d), dtype=bool)
    if len(d) > 1:
        last_of_grp[:-1] = new_grp[1:]
    last_of_grp[-1] = True
    d["_last_of_grp"] = last_of_grp
    inside = d[d["_active"]]

    events = inside.groupby("_event", sort=False, observed=True).agg(
        ticker=("ticker", "first"),
        date=("date", "first"),
        side=("Stack%", lambda s: int(np.sign(s.to_numpy()[0]))),
        start_dt=("dt", "first"),
        peak_abs=("Stack%", lambda s: float(np.abs(s.to_numpy()).max())),
        cut_by_day_end=("_last_of_grp", "last"),
    ).reset_index(drop=True)
    # an event is "normalized" only if it actually receded to exit_thr on its own --
    # not if the day simply ran out while it was still active.
    events["normalized"] = ~events["cut_by_day_end"]
    events = events.drop(columns=["cut_by_day_end"])

    start_time = events["start_dt"].astype(str).str.slice(11, 16)
    excluded = pd.Series(False, index=events.index)
    for lo, hi in ignore_windows:
        excluded |= (start_time >= lo) & (start_time <= hi)
    events = events[~excluded].reset_index(drop=True)
    return events


def build_ticker_stats(
    events: pd.DataFrame,
    modal_step: float = 0.1,
    modal_min_n: int = 3,
) -> pd.DataFrame:
    """Одна строка на тікер: total (усі події, обидва знаки), rating (частка
    подій, які СПРАВДІ нормалізувались до exit_thr, а не пік -- обрізана
    кінцем дня подія в рейтинг не рахується), peak_positive_pct /
    peak_negative_pct (модальний пік свого боку, окремо від рейтингу;
    від'ємний бік звітується як від'ємний %)."""
    if events.empty:
        return pd.DataFrame(columns=["ticker", "total", "rating", "peak_positive_pct", "peak_negative_pct"])

    ev = events.copy()
    total = ev.groupby("ticker", observed=True).size().rename("total")
    normalized_n = ev.groupby("ticker", observed=True)["normalized"].sum().rename("normalized_n")

    pos = ev[ev["side"] == 1]
    neg = ev[ev["side"] == -1]

    peak_pos = pos.groupby("ticker", observed=True)["peak_abs"].apply(
        lambda s: _modal_value(s.to_numpy(), step=modal_step, min_n=modal_min_n)
    ).rename("peak_positive_pct")
    peak_neg = neg.groupby("ticker", observed=True)["peak_abs"].apply(
        lambda s: _modal_value(s.to_numpy(), step=modal_step, min_n=modal_min_n)
    ).rename("peak_negative_pct")

    out = pd.concat([total, normalized_n, peak_pos, peak_neg], axis=1)
    out["total"] = out["total"].fillna(0).astype(int)
    out["normalized_n"] = out["normalized_n"].fillna(0).astype(int)
    out["rating"] = np.where(out["total"] > 0, out["normalized_n"] / out["total"], np.nan)
    out["peak_negative_pct"] = -out["peak_negative_pct"]  # звітуємо від'ємний бік як від'ємний %
    out = out.drop(columns=["normalized_n"]).reset_index()
    return out


## Збагачення: benchmark, ETF, lstcls, trash

- `bench` — беремо з самого `final.parquet` (останнє відоме значення на тікер).
- `ETF` та `lstcls` — обидва З ТЕЙПУ, тобто з файлів, які сам live-бридж уже
  пише на диск (`%APPDATA%\Axion	ape\intraday2\dateNy=...\`), а не з
  Datum-метаданих і не з CRACEN/`oc.parquet` (це був мій перший, гірший
  варіант — тут виправлено за прямою вказівкою). `IsETF` беремо з денного
  `static.parquet` (поле не змінюється впродовж дня, там найповніше
  покриття), `LstCls` — з кількох останніх `minuteIdx=.../part.parquet`
  (це Tier L поле, live, тому в `static.parquet` його немає — див.
  `docs/TAPE_FIELD_TIERS.md`). Це файловий read уже записаних даних, а не
  мережевий виклик до працюючого бриджа.
- `lstcls` — червоним, якщо < $5.
- `trash` — YES/червоним, якщо `lstcls < 5` (та сама межа, окремою колонкою
  для фільтрів). Примітка: сам тейп також має власне поле `IsCrap` — якщо
  malося на увазі саме воно (інша логіка, не $5), скажи, підключимо його
  замість цього порогу.

In [ ]:
def load_benchmarks(final_path) -> pd.DataFrame:
    tbl = pq.read_table(str(final_path), columns=["ticker", "bench"])
    df = tbl.to_pandas()
    df = df.dropna(subset=["bench"])
    last_bench = df.groupby("ticker", observed=True)["bench"].last()
    return last_bench.rename("bench").reset_index()


def load_tape_etf_and_lstcls(tape_root: Path, n_recent_minutes: int = 10) -> pd.DataFrame:
    """ETF та lstcls прямо з тейпу (файли, які вже пише живий бридж), не з
    Datum і не з CRACEN/oc.parquet. IsETF з денного static.parquet
    (найповніше покриття, поле незмінне впродовж дня). LstCls з кількох
    останніх part.parquet-хвилин того ж дня (Tier L, живе поле -- у
    static.parquet його нема). Повертає ['ticker','is_etf','lstcls']."""
    date_dirs = sorted(p for p in Path(tape_root).glob("dateNy=*") if p.is_dir())
    if not date_dirs:
        raise FileNotFoundError(f"No dateNy=* folders under {tape_root}")
    date_dir = date_dirs[-1]

    static_path = date_dir / "static.parquet"
    static_df = pd.read_parquet(static_path, columns=["Ticker", "IsETF"])
    static_df = static_df.rename(columns={"Ticker": "ticker", "IsETF": "is_etf"})
    static_df["ticker"] = static_df["ticker"].astype(str).str.upper().str.strip()
    static_df = static_df.drop_duplicates(subset=["ticker"], keep="last")

    minute_dirs = sorted(
        (p for p in date_dir.glob("minuteIdx=*") if p.is_dir()),
        key=lambda p: int(p.name.split("=", 1)[1]),
    )
    frames = []
    for p in minute_dirs[-n_recent_minutes:]:
        part = p / "part.parquet"
        if part.exists():
            frames.append(pd.read_parquet(part, columns=["Ticker", "LstCls", "IsETF"]))
    if not frames:
        raise FileNotFoundError(f"No minuteIdx=*/part.parquet under {date_dir}")
    tape = pd.concat(frames, ignore_index=True)
    tape = tape.rename(columns={"Ticker": "ticker", "LstCls": "lstcls", "IsETF": "is_etf_minute"})
    tape["ticker"] = tape["ticker"].astype(str).str.upper().str.strip()
    tape = tape.dropna(subset=["lstcls"])
    minute_agg = (
        tape.sort_values(["ticker"])
        .groupby("ticker", observed=True)
        .last()[["lstcls", "is_etf_minute"]]
        .reset_index()
    )

    out = static_df.merge(minute_agg[["ticker", "lstcls"]], on="ticker", how="outer")
    out = out.merge(minute_agg[["ticker", "is_etf_minute"]], on="ticker", how="left")
    out["is_etf"] = out["is_etf"].where(out["is_etf"].notna(), out["is_etf_minute"])
    return out[["ticker", "is_etf", "lstcls"]]


In [ ]:
def build_peak_deviation_report(
    final_path=FINAL_PATH,
    tape_root=TAPE_ROOT,
    start_thr: float = 0.5,
    exit_thr: float = 0.1,
    min_total: int = 10,
    min_rating: float = 0.6,
    ignore_windows=(("03:58", "04:02"), ("07:58", "08:02")),
    lstcls_penny_threshold: float = 5.0,
) -> pd.DataFrame:
    print(f"Reading Stack% series -> {final_path}", flush=True)
    tbl = pq.read_table(str(final_path), columns=["ticker", "date", "dt", "Stack%"])
    df = tbl.to_pandas()

    events = detect_events(df, start_thr=start_thr, exit_thr=exit_thr, ignore_windows=ignore_windows)
    del df
    print(f"events: {len(events):,} across {events['ticker'].nunique():,} tickers", flush=True)

    stats = build_ticker_stats(events)

    bench = load_benchmarks(final_path)
    stats = stats.merge(bench, on="ticker", how="left")

    tape = load_tape_etf_and_lstcls(tape_root)
    stats = stats.merge(tape, on="ticker", how="left")

    def _etf_label(v):
        if pd.isna(v):
            return ""
        return "YES" if bool(v) else "NO"

    stats["ETF"] = stats["is_etf"].apply(_etf_label)
    stats["trash"] = np.where(stats["lstcls"] < lstcls_penny_threshold, "YES", "NO")

    out = stats[(stats["total"] >= min_total) & (stats["rating"] >= min_rating)].copy()
    out = out.sort_values(["rating", "total"], ascending=[False, False]).reset_index(drop=True)

    cols = ["ticker", "bench", "rating", "total", "peak_positive_pct", "peak_negative_pct", "ETF", "lstcls", "trash"]
    out = out[cols]
    out["rating"] = out["rating"].round(3)
    out["peak_positive_pct"] = out["peak_positive_pct"].round(3)
    out["peak_negative_pct"] = out["peak_negative_pct"].round(3)
    out["lstcls"] = out["lstcls"].round(2)
    return out


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font

GREEN_FILL = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
RED_FILL = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
RED_FONT = Font(color="9C0006")


def write_colored_report(df: pd.DataFrame, out_path, lstcls_penny_threshold: float = 5.0) -> str:
    """Пише звіт у .xlsx з розфарбуванням: ETF NO=зелений/YES=червоний,
    lstcls червоним якщо < межі, trash YES=червоний/NO=зелений."""
    wb = Workbook()
    ws = wb.active
    ws.title = "peak_deviation"

    ws.append(list(df.columns))
    for row in df.itertuples(index=False):
        ws.append([None if pd.isna(v) else v for v in row])

    col_idx = {name: i + 1 for i, name in enumerate(df.columns)}
    n_rows = len(df)

    for r in range(2, n_rows + 2):
        if "ETF" in col_idx:
            cell = ws.cell(row=r, column=col_idx["ETF"])
            cell.fill = RED_FILL if str(cell.value).upper() == "YES" else GREEN_FILL
        if "trash" in col_idx:
            cell = ws.cell(row=r, column=col_idx["trash"])
            cell.fill = RED_FILL if str(cell.value).upper() == "YES" else GREEN_FILL
        if "lstcls" in col_idx:
            cell = ws.cell(row=r, column=col_idx["lstcls"])
            v = cell.value
            if v is not None and v < lstcls_penny_threshold:
                cell.font = RED_FONT
                cell.fill = RED_FILL

    for i, name in enumerate(df.columns, start=1):
        width = max(10, min(24, int(df[name].astype(str).str.len().max() if len(df) else 10) + 2))
        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = width

    wb.save(str(out_path))
    return str(out_path)


In [ ]:
report = build_peak_deviation_report(FINAL_PATH, TAPE_ROOT)
print(f"Тікерів у звіті (rating>=0.6, total>=10): {len(report):,}")
report.head(20)


In [ ]:
xlsx_path = OUT_DIR / "peak_deviation.xlsx"
write_colored_report(report, xlsx_path)
print("Записано:", xlsx_path)
